### Before ORF prediction

In [17]:
import pandas as pd
import re
import numpy as np

df = pd.read_csv('paired_refseq_viral_and_astro_orfs_1_50_batches_results.tsv', sep='\t')

C:\Users\Mate_\AppData\Local\Temp\ipykernel_25232\3894979438.py:5: DtypeWarning: Columns (1: Страна) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('paired_refseq_viral_and_astro_orfs_1_50_batches_results.tsv', sep='\t')


In [ ]:
def is_astrovirus(blast_hit):
    if pd.isna(blast_hit) or str(blast_hit).strip() == "":
        return False
    hit = str(blast_hit).lower()
    astro_patterns = ['astrovirus', 'astroviridae', 'mamastrovirus', 'avastrovirus']
    for pattern in astro_patterns:
        if re.search(r'\b' + pattern, hit, re.IGNORECASE):
            return True
    return False

def _orf_from_blast_hit(blast_hit):
    if pd.isna(blast_hit) or str(blast_hit).strip() == "":
        return None

    hit = str(blast_hit)

    if re.search(r'\b1AB\b', hit, re.IGNORECASE) or \
       re.search(r'polyprotein\s+1AB', hit, re.IGNORECASE):
        return '1AB'

    if re.search(r'\bORF\s*1A\b', hit, re.IGNORECASE):
        return '1A'

    if re.search(r'\bORF\s*1B\b', hit, re.IGNORECASE):
        return '1B'

    if re.search(r'\bORF\s*2\b', hit, re.IGNORECASE) or \
       re.search(r'\bcapsid\b', hit, re.IGNORECASE) or \
       re.search(r'\bVP90\b', hit, re.IGNORECASE):
        return '2'

    return None


def analyze_orf_types(top5_profiles, contig_len=None, blast_hit=None):
    if pd.isna(top5_profiles) or str(top5_profiles).strip() == "":
        return set()

    profiles = str(top5_profiles)
    orf_types = set()

    for part in profiles.split(';'):
        part = part.strip()
        part = re.sub(r'\s*\([^)]*\)', '', part)

        is_astro = False
        astro_patterns = ['astroviridae', 'astrovirus', 'mamastrovirus']
        for pattern in astro_patterns:
            if re.search(pattern, part, re.IGNORECASE):
                is_astro = True
                break

        if not is_astro:
            continue

        if re.search(r'1A\b', part, re.IGNORECASE):
            orf_types.add('1A')

        if re.search(r'1B\b', part, re.IGNORECASE) or \
           re.search(r'1b_pcr_primer_region', part, re.IGNORECASE):
            orf_types.add('1B')

        if re.search(r'\b2\b', part) or \
           re.search(r'capsid', part, re.IGNORECASE) or \
           re.search(r'VP90', part):
            orf_types.add('2')

        if re.search(r'1ab\b', part, re.IGNORECASE):
            if '1A' not in orf_types and '1B' not in orf_types:
                orf_types.add('1A')

    if len(orf_types) > 1:
        try:
            length_val = float(contig_len)
        except (TypeError, ValueError):
            length_val = None

        if length_val is not None and length_val < 1000:
            chosen = _orf_from_blast_hit(blast_hit)

            if chosen is None:
                chosen = sorted(orf_types)[0]

            if chosen == '1AB':
                chosen = '1A' if '1A' in orf_types else '1A'

            orf_types = {chosen}

    return orf_types

df['is_astrovirus'] = (df['Бласт хит'].apply(is_astrovirus) & df['Профиль'].apply(is_astrovirus))
df['orf_types'] = df.apply(
    lambda row: analyze_orf_types(
        row['Топ 5 профилей'],
        contig_len=row['Длина контига, нт'],
        blast_hit=row['Бласт хит'],
    ),
    axis=1,
)
df['orf_count'] = df['orf_types'].apply(len)

df['genome_type'] = df['orf_types'].apply(
    lambda x: 'Полный геном (3 ORFs)' if x == {'1A', '1B', '2'} 
    else 'Неполный геном' if x 
    else 'Нет ORFs'
)

astro_df = df[df['is_astrovirus'] == True].copy()

print(f"Найдено {len(astro_df)} записей, относящихся к астровирусам")
print(f"Это составляет {len(astro_df)/len(df)*100:.2f}% от всех записей")
print(f"Уникальных образцов с астровирусами: {astro_df['Образец'].nunique()}")

Найдено 1229 записей, относящихся к астровирусам
Это составляет 0.86% от всех записей
Уникальных образцов с астровирусами: 176


In [3]:
astro_df.to_csv('astrovirus_contigs.csv', index=False)

In [ ]:
from collections import Counter, defaultdict

genome_counts = astro_df['genome_type'].value_counts()
print("\nРаспределение по типам геномов:")
for gtype, count in genome_counts.items():
    print(f"  {gtype}: {count} ({count/len(astro_df)*100:.2f}%)")

orf_count_dist = astro_df['orf_count'].value_counts().sort_index()
print("\nРаспределение по количеству ORFs:")
for count, num in orf_count_dist.items():
    print(f"  {count} ORF: {num} ({num/len(astro_df)*100:.2f}%)")

all_orf_types = []
for orf_set in astro_df['orf_types']:
    all_orf_types.extend(orf_set)
orf_type_counts = Counter(all_orf_types)
print("\nТипы ORFs:")
for orf_type, count in orf_type_counts.most_common():
    print(f"  {orf_type}: {count} ({count/len(astro_df)*100:.2f}%)")

СТАТИСТИКА ПО АСТРОВИРУСАМ

Распределение по типам геномов:
  Неполный геном: 1192 (96.99%)
  Полный геном (3 ORFs): 37 (3.01%)

Распределение по количеству ORFs:
  1 ORF: 1153 (93.82%)
  2 ORF: 39 (3.17%)
  3 ORF: 37 (3.01%)

Типы ORFs:
  1A: 664 (54.03%)
  1B: 355 (28.89%)
  2: 323 (26.28%)


In [6]:
import re

pairs = set()
for _, row in astro_df.iterrows():
    sample = str(row["Образец"]).strip()
    info = str(row["Инфо"])
    m = re.search(r"Contig_ID:\s*(\S+)", info)
    if m:
        pairs.add((sample, m.group(1).strip()))

print(f"Пар (образец, contig) в таблице: {len(pairs)}")

fasta_in = "all_assembled_contigs.fasta"
fasta_out = "all_astro_contigs.fasta"

kept = 0
total = 0
found_pairs = set()

with open(fasta_in, encoding='utf-8') as fin, open(fasta_out, 'w', encoding='utf-8') as fout:
    write_record = False
    for line in fin:
        if line.startswith(">"):
            total += 1
            header = line[1:].strip()
            first_part = header.split("/", 1)[0]      
            parts = first_part.split("|")
            if len(parts) >= 2:
                key = (parts[0], parts[1])             
                if key in pairs:
                    write_record = True
                    kept += 1
                    found_pairs.add(key)
                    fout.write(line)
                else:
                    write_record = False
            else:
                write_record = False
        else:
            if write_record:
                fout.write(line)

print(f"Всего записей в FASTA: {total}")
print(f"Оставлено: {kept}")

missing = pairs - found_pairs
if missing:
    print(f"\nНе нашлось в FASTA пар: {len(missing)}")
    for s, c in list(missing)[:20]:
        print(f"  {s} | {c}")
else:
    print("\nВсе пары из таблицы найдены в FASTA.")

Пар (образец, contig) в таблице: 1229
Всего записей в FASTA: 148964
Оставлено: 1229

Все пары из таблицы найдены в FASTA.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.ticker import MaxNLocator

def fasta_lengths(path):
    lengths, n = [], 0
    with open(path) as fh:
        for line in fh:
            if line.startswith(">"):
                if n: lengths.append(n)
                n = 0
            else:
                n += len(line.strip())
    if n: lengths.append(n)
    return lengths

path = "all_astro_contigs.fasta"
lengths = np.array(fasta_lengths(path), dtype=float)

fig, ax = plt.subplots(figsize=(14, 6))
ax.hist(lengths, bins=200, color="steelblue", edgecolor="black", alpha=0.8)
ax.set_title(f"Распределение длин астровирусных контигов "
             f"(N={len(lengths)})")
ax.set_xlabel("Длина последовательности, нт")
ax.set_ylabel("Количество")

ax.set_xticks(np.arange(0, lengths.max(), 500))
ax.yaxis.set_major_locator(MaxNLocator(integer=True))
plt.xticks(rotation=0)

plt.tight_layout()
plt.show()

### After ORF prediction

In [ ]:
import re
import pandas as pd
from Bio import SeqIO

INP = "orfs_1B.faa"
MIN_LEN = 100
MAX_LEN = 600
OUT_FILTERED = "orfs_1B_min_100_aa.faa"
OUT_FINAL = "orfs_1B_clean.faa"
META_TSV = "paired_refseq_viral_and_astro_orfs_1_50_batches_results.tsv"

df = pd.read_csv(META_TSV, sep="\t", dtype=str)
df["_contig"] = df["Инфо"].map(
    lambda x: (re.search(r"Contig_ID:\s*([^|]+)", x or "") or [None, None])[1]
)
df["_contig"] = df["_contig"].str.strip()
lookup = {(r["Образец"], r["_contig"]): r for _, r in df.iterrows()}

filtered = [
    rec for rec in SeqIO.parse(INP, "fasta")
    if MIN_LEN <= len(rec.seq) <= MAX_LEN
]
SeqIO.write(filtered, OUT_FILTERED, "fasta")
print(f"{OUT_FILTERED}: {len(filtered)} seqs kept ({MIN_LEN} <= len <= {MAX_LEN} aa)")

records = []
for rec in SeqIO.parse(OUT_FILTERED, "fasta"):
    m = re.match(r"^([^|]+)\|([^/]+)", rec.description)
    row = lookup.get((m.group(1), m.group(2))) if m else None
    if row is not None:
        new = f"{m.group(1)}/{m.group(2)}/{row['Страна']}/{row['Год']}/{row['Хозяин']}"
        rec.description = new
        rec.id = new.split()[0]
    rec.description = rec.description.replace(" ", "-")
    rec.id = rec.description.split()[0]
    records.append(rec)

SeqIO.write(records, OUT_FINAL, "fasta")
print(f"Записано {len(records)} записей в {OUT_FINAL}")

orfs_1B_min_100_aa.faa: 226 seqs kept (100 <= len <= 600 aa)
Записано 226 записей в orfs_1B_min_100_aa_clean.faa


### Get GenBank centroids + merge with contigs

In [9]:
import re
import pandas as pd
from Bio import SeqIO

BAT_GENERA = {
    "Miniopterus", "Nyctalus", "Eidolon", "Rousettus",
    "Myotis", "Hipposideros", "Rhinolophus", "Scotophilus",
    "Pipistrellus", "Tylonycteris", "Mops", "Coleura",
    "Pteropus", "Eptesicus", "Tadarida", "Molossus", "Carollia",
    "Artibeus", "Desmodus", "Glossophaga", "Anoura", "Megaderma",
    "Rhinopoma", "Mormopterus", "Chaerephon", "Nycteris",
    "Emballonura", "Taphozous", "Saccolaimus", "Chalinolobus",
    "Lasiurus", "Nycticeius", "Plecotus", "Barbastella", "Murina",
    "Kerivoula", "Phoniscus", "Cynopterus", "Macroglossus",
    "Eonycteris", "Epomophorus", "Epomops", "Hypsignathus",
    "Micropteropus", "Myonycteris", "Lissonycteris",
}

_BAT_RE = re.compile(
    r'\b(' + '|'.join(re.escape(g) for g in sorted(BAT_GENERA, key=len, reverse=True)) + r')\b',
    re.IGNORECASE,
)
_CHIROPTERA_RE = re.compile(r'\bChiroptera\b', re.IGNORECASE)
_BAT_WORD_RE = re.compile(r'\b(bat|bats|flying[\s-]?fox|fruit[\s-]?bat)\b', re.IGNORECASE)

def is_bat(description: str) -> bool:
    return bool(
        _CHIROPTERA_RE.search(description)
        or _BAT_RE.search(description)
        or _BAT_WORD_RE.search(description)
    )

# --- Читаем таблицу ---
df = pd.read_csv(
    "Astroviridae_complete_1B_17nt_mmseq2.tsv",
    sep="\t",
    header=None,
    names=["center", "member"],
    dtype=str,
)

def acc(x):
    return x.split("/")[0].strip()

counts = df["center"].map(acc).value_counts().to_dict()
center_accs = set(counts.keys())

# 1) Описание каждого центроида (первое встреченное)
center_desc = {}
for c in df["center"]:
    a = acc(c)
    center_desc.setdefault(a, c)

# 2) Какие центроиды — летучие мыши (по описанию ЦЕНТРОИДА)
bat_centers = {a for a, desc in center_desc.items() if is_bat(desc)}
nonbat_centers = center_accs - bat_centers

# 3) Акцессии, относящиеся к кластерам летучих центроидов:
#    сам центроид + все его члены
bat_cluster_accs = set(bat_centers)
for _, row in df.iterrows():
    c_acc = acc(row["center"])
    m_acc = acc(row["member"])
    if c_acc in bat_centers:
        bat_cluster_accs.add(c_acc)
        bat_cluster_accs.add(m_acc)

# --- Проходим по FASTA ---
records = []
n_bat_cluster = 0
n_nonbat_centers = 0

for rec in SeqIO.parse("astro_complete_1b_genbank_mammalia.faa", "fasta"):
    acc_id = rec.id.split("|")[0]

    if acc_id in bat_cluster_accs:
        # Кластер летучей мыши — берём всё, без /N
        records.append(rec)
        n_bat_cluster += 1
    elif acc_id in nonbat_centers:
        # Не-летучий центроид — только он, с /N
        n = counts[acc_id]
        new = f"{rec.description}/{n}"
        rec.description = new
        rec.id = new.split()[0]
        records.append(rec)
        n_nonbat_centers += 1
    # остальные (члены не-летучих кластеров) — пропускаем

SeqIO.write(records, "genbank_centers_and_all_bats.faa", "fasta")
print(f"Записано всего: {len(records)}")
print(f"  записей из кластеров летучих мышей: {n_bat_cluster}")
print(f"  не-летучих центроидов:                {n_nonbat_centers}")

Записано всего: 148
  записей из кластеров летучих мышей: 54
  не-летучих центроидов:                94


In [10]:
def merge_fasta(file1, file2, output):
    with open(output, 'w') as out:
        for fname in (file1, file2):
            with open(fname) as f:
                for line in f:
                    out.write(line)
    print(f"Merged into {output}")

merge_fasta("astro_complete_1b_genbank_mammalia_centers.faa", "orfs_1B_clean.faa", "genbank_and_contigs_1b.fasta")

Merged into genbank_and_contigs_1b.fasta


### After alignment, remove empty and short from pcr fragment

In [11]:
from Bio import AlignIO
from Bio.Align import MultipleSeqAlignment
import os

INPUT_FILE  = "genbank_and_contigs_pcr_aln.fasta"
OUTPUT_FILE = "genbank_and_contigs_pcr_aln_filtered.fasta"
MIN_LENGTH  = 100

ext = os.path.splitext(INPUT_FILE)[1].lower()
fmt = {".fasta": "fasta", ".fa": "fasta", ".faa": "fasta",
       ".aln": "clustal", ".clustal": "clustal",
       ".phy": "phylip", ".sto": "stockholm"}.get(ext, "fasta")

aln = AlignIO.read(INPUT_FILE, fmt)

gap_chars = set("-.")
kept = [r for r in aln
        if sum(1 for c in str(r.seq) if c not in gap_chars and c != " ") >= MIN_LENGTH]

AlignIO.write(MultipleSeqAlignment(kept), OUTPUT_FILE, "fasta")

1